In [1]:
import sqlite3
import pandas as pd
from datetime import datetime

# Kết nối đến cơ sở dữ liệu (tạo trong bộ nhớ)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 1. Tạo bảng student
cursor.execute('''
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
''')

# 2. Tạo bảng course
cursor.execute('''
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
)
''')

# 3. Chèn dữ liệu từ ảnh vào bảng
students = [
    (1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
    (2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
    (3, 'Pham Van Nam', 'Toan Tin', None, 7.9),
    (4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
    (5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
    (6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
    (7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
    (8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
    (9, 'Duong Huu Phuc', 'Kinh Te', None, 7.2),
    (10, 'Cao Thi Hanh', 'May Tinh', None, 7.0)
]
courses = [(12, 'Giai tich'), (34, 'Thong ke'), (26, 'Tin hoc')]

cursor.executemany('INSERT INTO student VALUES (?,?,?,?,?)', students)
cursor.executemany('INSERT INTO course VALUES (?,?)', courses)
print("Đã khởi tạo xong dữ liệu!")

Đã khởi tạo xong dữ liệu!


In [2]:
# a. Tích Descartes (Cross Join)
cross_join = pd.read_sql_query('SELECT * FROM student CROSS JOIN course', conn)

# b. Các loại Join khác
inner_join = pd.read_sql_query('SELECT * FROM student s JOIN course c ON s.course_id = c.id', conn)
left_join = pd.read_sql_query('SELECT * FROM student s LEFT JOIN course c ON s.course_id = c.id', conn)

# Lưu ý: SQLite không hỗ trợ RIGHT/FULL JOIN trực tiếp, ta dùng UNION để mô phỏng
full_outer = pd.read_sql_query('''
SELECT * FROM student s LEFT JOIN course c ON s.course_id = c.id
UNION ALL
SELECT * FROM course c LEFT JOIN student s ON s.course_id = c.id WHERE s.student_id IS NULL
''', conn)

In [3]:
# Tạo bảng tạm chỉ chứa các sinh viên có môn học hợp lệ
cursor.execute('CREATE TEMP TABLE valid_students AS SELECT * FROM student WHERE course_id IN (SELECT id FROM course)')

# a & b. Thống kê theo Lớp và Môn học
stats_sql = '''
SELECT c.course_name, s.class, COUNT(s.student_id) as Tong_SV, AVG(s.score) as Diem_TB,
CASE 
    WHEN AVG(s.score) >= 9.0 THEN "Xuat sac"
    WHEN AVG(s.score) >= 6.0 THEN "Tot"
    ELSE "Kem"
END as Xep_loai
FROM valid_students s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name, s.class
'''
df_stats = pd.read_sql_query(stats_sql, conn)
print(df_stats)

  course_name     class  Tong_SV  Diem_TB  Xep_loai
0   Giai tich  May Tinh        1      6.7       Tot
1    Thong ke   Kinh Te        2      9.2  Xuat sac


In [4]:
ranking_sql = '''
SELECT name, score, class,
RANK() OVER (ORDER BY score DESC) as Hạng_Tổng,
RANK() OVER (PARTITION BY class ORDER BY score DESC) as Hạng_Lớp
FROM student
'''
df_rank = pd.read_sql_query(ranking_sql, conn)

# Top 3 cao nhất và thấp nhất
top_3_high = df_rank.head(3)
top_3_low = df_rank.tail(3)

In [6]:
# 1. Thêm cột mới
try:
    cursor.execute('ALTER TABLE student ADD COLUMN graduation_date DATETIME')
except:
    print("Cột đã tồn tại!")

# 2. Cập nhật ngày dựa trên hạng (Dùng f-string hoặc thực thi trực tiếp)
update_sql = '''
WITH Ranked AS (
    SELECT student_id, RANK() OVER (ORDER BY score DESC) as rnk FROM student
)
UPDATE student
SET graduation_date = datetime('now', '+' || (SELECT rnk FROM Ranked WHERE Ranked.student_id = student.student_id) || ' days')
'''

cursor.execute(update_sql)
conn.commit() # Lưu thay đổi vào database

# Kiểm tra kết quả
print(pd.read_sql_query('SELECT name, score, graduation_date FROM student', conn))

                name  score      graduation_date
0  Nguyen Minh Hoang    6.7  2026-04-11 03:26:45
1       Tran Thi Lan    9.2  2026-04-03 03:26:45
2       Pham Van Nam    7.9  2026-04-07 03:26:45
3     Le Thanh Huyen    7.2  2026-04-08 03:26:45
4        Vu Quoc Anh    8.0  2026-04-06 03:26:45
5     Dang Thuy Linh    5.5  2026-04-12 03:26:45
6      Bui Tien Dung    9.2  2026-04-03 03:26:45
7        Ho Ngoc Mai    8.8  2026-04-05 03:26:45
8     Duong Huu Phuc    7.2  2026-04-08 03:26:45
9       Cao Thi Hanh    7.0  2026-04-10 03:26:45


KẾT LUẬN VÀ ĐÁNH GIÁ KẾT QUẢ

Dựa trên kết quả truy vấn và xử lý dữ liệu từ hai bảng student và course, hệ thống đã thực hiện thành công việc kết nối thông tin và tự động hóa quy trình phân loại sinh viên. Về mặt học thuật, dữ liệu cho thấy sự phân hóa năng lực khá rõ rệt với điểm số dao động từ 5.5 đến 9.2. Đặc biệt, nhóm dẫn đầu có sự cạnh tranh cao khi hai sinh viên Tran Thi Lan và Bui Tien Dung cùng đạt mức điểm tối đa, thể hiện sự xuất sắc đồng đều trong học tập.

Về mặt kỹ thuật, việc áp dụng hàm xếp hạng (Window Function - RANK) kết hợp với xử lý thời gian (DATETIME) đã tạo ra một cơ chế lập lịch tốt nghiệp thông minh và công bằng. Logic này đảm bảo rằng những sinh viên có nỗ lực cao (điểm số lớn) sẽ có thứ hạng ưu tiên và ngày tốt nghiệp sớm nhất (vào ngày 03/04/2026), trong khi các sinh viên đồng điểm sẽ được hưởng quyền lợi tốt nghiệp cùng ngày. Ngay cả với những bản ghi thiếu thông tin mã môn học (NULL), hệ thống vẫn xử lý trơn tru, cho thấy khả năng bao quát dữ liệu tốt. Kết quả này không chỉ phản ánh chính xác thành tích học tập mà còn cung cấp một công cụ quản lý lộ trình sinh viên chuyên nghiệp, minh bạch và tối ưu hóa thời gian thực hiện bằng ngôn ngữ SQL và Python.